# IRIS baseline measurements

Use a GPU runtime and stop previous qualification servers. Start with the batching/phase audit. Compare standalone and HTTP scopes separately; these baselines keep FP32 and compilation disabled. The T4 batching/phase audit passed at 4b8db96e: eager selected multirow worker steps, while captured used single rows. The standalone GPU and uninstrumented timing sweep passed at b16393e6. Results and ranges are in docs/design/iris-t4-benchmark.md; rerun on your checkout to reproduce. See [README](https://github.com/Artifex2002/mstar/blob/codex/iris/benchmark/iris/README.md).


In [ ]:
%pip install -q uv


In [ ]:
%%bash
set -euo pipefail
if [ ! -e /content/mstar-iris ]; then
  git clone --branch codex/iris --single-branch https://github.com/Artifex2002/mstar.git /content/mstar-iris
fi
cd /content/mstar-iris
git switch codex/iris
git pull --ff-only origin codex/iris
if [ ! -e .venv ]; then
  uv venv --python 3.12 .venv
fi
export PATH="$HOME/.cargo/bin:$PWD/.venv/bin:$PATH"
if ! command -v rustup >/dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
uv pip install --python .venv/bin/python --torch-backend=auto \
  -r scripts/iris/requirements-cuda-tests.txt -e '.[iris,metrics]' maturin
env -u CONDA_PREFIX VIRTUAL_ENV="$PWD/.venv" \
  .venv/bin/maturin develop --release --locked -m rust/Cargo.toml
.venv/bin/python -c 'import mstar_rust; print(mstar_rust.__version__)'

mkdir -p .ci-results/iris
if [ ! -e .ci-results/iris/reference ]; then
  git clone https://github.com/eloialonso/iris.git .ci-results/iris/reference
fi
git -C .ci-results/iris/reference checkout --detach 24326aaaa283c527f42b89b44cfdecf2665a7a16

if ! command -v ss >/dev/null 2>&1; then
  apt-get -qq update
  apt-get -qq install -y iproute2
fi


## 1. Verify worker batching and phases

Send this output first. Missing histograms fail; a pass with only single-row steps is still an observation to investigate before claiming a batching gain.


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
source .venv/bin/activate
mkdir -p .ci-results/iris
OMP_NUM_THREADS=2 python -m benchmark.iris.run --stage audit \
  --download-checkpoint --output .ci-results/iris/benchmark-audit-02 \
  2>&1 | tee .ci-results/iris/benchmark-audit-02.log


## 2. Full measurement sweep

Run after the audit is understood. The timings stage skips the completed audit. Three rounds interleave official PyTorch / eager / captured standalone arms and eager / captured HTTP arms, with fresh processes, warmup and fixed workloads. This can take many minutes. Use new directory names on reruns and update the download cell.


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
source .venv/bin/activate
mkdir -p .ci-results/iris
OMP_NUM_THREADS=2 python -m benchmark.iris.run --stage timings \
  --download-checkpoint --rounds 3 --samples 5 --requests 10 --warmup-waves 5 \
  --output .ci-results/iris/benchmark-01 \
  2>&1 | tee .ci-results/iris/benchmark-01.log


## Download available evidence, including on failure


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

root = Path("/content/mstar-iris/.ci-results/iris")
archive = Path("/content/iris-benchmark-01.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for name in ("benchmark-audit-02", "benchmark-01"):
        directory = root / name
        if directory.exists():
            for path in sorted(directory.rglob("*")):
                if path.is_file() and "inductor" not in path.parts:
                    bundle.write(path, path.relative_to(root))
        log = root / (name + ".log")
        if log.exists():
            bundle.write(log, log.relative_to(root))
files.download(str(archive))
